In [ ]:
import os

sys.path.append('/mnt/c/Users/Isabella/tcc')
DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/david'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'david_gt.txt')
output_dir = "resultados_video/"
video_path = "resultados_video/tracking_output.mp4"

In [ ]:
import cv2
import glob
import numpy as np
from wisardpkg import ClusWisard
import sys
from wisard_object_tracker.src.utils import tracker_utils
import matplotlib.pyplot as plt
import json

In [ ]:
# Carrega imagens
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truths
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

In [ ]:
class AdaptiveBackgroundModel:
    def __init__(self, alpha=0.03):
        self.alpha = alpha
        self.background = None

    def initialize(self, frame):
        self.background = frame.astype(np.float32)

    def update(self, frame, learn_mask):
        if self.background is None:
            self.initialize(frame)
            return

        bg = self.background
        fg = frame.astype(np.float32)

        mask = learn_mask.astype(bool)
        bg[mask] = (1 - self.alpha) * bg[mask] + self.alpha * fg[mask]

        self.background = bg

    def subtract(self, frame, diff_threshold=30):
        bg_uint8 = self.background.astype(np.uint8)
        diff = cv2.absdiff(frame, bg_uint8)
        gray = cv2.cvtColor(diff, cv2.COLOR_RGB2GRAY)
        _, fg_mask = cv2.threshold(
            gray, diff_threshold, 255, cv2.THRESH_BINARY
        )
        return fg_mask
def adaptive_background_subtraction(
    frame,
    prev_bbox,
    bg_model,
    tracking_confidence,
    confidence_threshold,
    expand_ratio=0,
    diff_threshold=10,
):
    h, w = frame.shape[:2]
    x, y, bw, bh = map(int, prev_bbox)

    pad_w = int(bw * expand_ratio)
    pad_h = int(bh * expand_ratio)

    x0 = max(0, x - pad_w)
    y0 = max(0, y - pad_h)
    x1 = min(w, x + bw + pad_w)
    y1 = min(h, y + bh + pad_h)

    # -----------------------------
    # 1️⃣ Subtrai fundo primeiro
    # -----------------------------
    fg_mask = bg_model.subtract(frame, diff_threshold)

    # Preserva objeto
    fg_mask[y0:y1, x0:x1] = 255
    foreground = cv2.bitwise_and(frame, frame, mask=fg_mask)

    # -----------------------------
    # 2️⃣ Aprende fundo SOMENTE se tracking for confiável
    # -----------------------------
    if tracking_confidence >= confidence_threshold:
        learn_mask = np.ones((h, w), dtype=np.uint8) * 255
        learn_mask[y0:y1, x0:x1] = 0
        bg_model.update(frame, learn_mask)
    else:
        # fundo congelado — evita drift
        pass

    return foreground


# Primeiro frame

In [ ]:
# Carrega primeiro frame  # lembrar de carregar certo em tons de cinza
first_frame = cv2.imread(image_paths[0])
print(f"Primeiro frame carregado: {first_frame.shape}")

first_gt = all_ground_truths[0]
print(f"Ground Truth do primeiro frame: {first_gt}")

# Mostra o frame com a bbox
first_frame_with_bbox = first_frame.copy()
x, y, w, h = map(int, first_gt)
cv2.rectangle(first_frame_with_bbox, (x, y), (x + w, y + h), (0, 255, 0), 2) 

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(first_frame_with_bbox, cv2.COLOR_BGR2RGB))
plt.title('Primeiro Frame - Objeto a Rastrear')
plt.axis('off')
plt.show()

In [ ]:
# --- PASSO 2: EXTRAIR E PRÉ-PROCESSAR PATCH ---
print("\n Extraindo e pré-processando patch...")

# Extrai patch do objeto
patch = tracker_utils.extract_patch(first_frame, first_gt)

# Mostra o patch ORIGINAL (antes do pré-processamento)
plt.figure(figsize=(8, 6))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
plt.title(f'Patch Original\n shape:{patch.shape}')
plt.axis('off')


In [ ]:
def preprocess_patch(patch):
    # Converte para grayscale caso seja RGB
    if len(patch.shape) == 3:
        gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
    else:
        gray = patch.copy()

    # 1) MÉDIA GLOBAL  (correção do erro)
    global_mean = int(np.mean(gray))
    no_bg_global = cv2.subtract(gray, global_mean)

    # 2) Binarização Otsu
    _, otsu = cv2.threshold(
        no_bg_global, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # 3) Retornar vetor flatten binário
    vector = (otsu.flatten() > 0).astype(np.uint8)

    return vector


# def preprocess_patch(patch, show=True):
#     if len(patch.shape) == 3:
#         gray = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
#     else:
#         gray = patch.copy()
    
#     global_mean = int(np.mean(gray))
#     no_bg_global = cv2.subtract(gray, global_mean)
#     # Otsu adaptativo
#     otsu_local = cv2.adaptiveThreshold(
#         no_bg_global, 255,
#         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
#         cv2.THRESH_BINARY,
#         9, 9
#     )
    
#     # 3) Retornar vetor flatten binário
#     vector = (otsu_local.flatten() > 0).astype(np.uint8)
    
#     return vector

In [ ]:
first_pattern = preprocess_patch(patch)
print(f"Primeiro Pattern pré-processado: {first_pattern.shape}")

In [ ]:
# Parâmetros
parameters = {
    # "INPUT_PATTERN_SIDE": 32,                  
    "CLUSWISARD_ADDRESS_SIZE":7,          
    "MAX_RETRAIN_SCORE": 0.5,       
    # "MIN_RETRAIN_SCORE": 0.4,    
    "CLUSWISARD_MIN_SCORE": 0.35,               
    "CLUSWISARD_THRESHOLD": 1,                 
    "CLUSWISARD_DISCRIMINATOR_LIMIT": 4,      
    "CLUSWISARD_BLEACHING_ACTIVATED": True,     
    "CLUSWISARD_ACTIVATION_DEGREE": True,
    "CLUSWISARD_RETURN_CONFIDENCE": True,
    "CLUSWISARD_CLASSES_DEGREES": True,
    "MAX_SEARCH_WINDOW_SCALE": 0.2,
    "STEP_SIZE": 5,
}

clus = ClusWisard(
            parameters['CLUSWISARD_ADDRESS_SIZE'], # address size
            parameters['CLUSWISARD_MIN_SCORE'], # min score
            parameters['CLUSWISARD_THRESHOLD'], # threshold
            parameters['CLUSWISARD_DISCRIMINATOR_LIMIT'], # discriminator limit
            bleachingActivated=parameters['CLUSWISARD_BLEACHING_ACTIVATED'],
            returnActivationDegree=parameters['CLUSWISARD_ACTIVATION_DEGREE'],
            returnConfidence=parameters['CLUSWISARD_RETURN_CONFIDENCE'],
            returnClassesDegrees=parameters['CLUSWISARD_CLASSES_DEGREES']
        )


In [ ]:
# Treina com o patch do objeto
clus.train([first_pattern.tolist()], ["object"])
print("ClusWisard treinado com o patch do primeiro frame!")

In [ ]:
print("\n Testando autorreconhecimento...")

# Classifica o MESMO patch usado no treinamento
result = clus.classify([first_pattern.tolist()])[0]
activation = result.get("activationDegree", -1)

print(f"Ativação com o próprio patch de treinamento: {activation}")
print(result)

# Região de Busca

In [ ]:
def generate_search_regions_circular(
    prev_bbox, 
    frame_shape, 
    search_region_scale=1, 
    step_size=0.5,
    start_angle=0,
    clockwise=True
):
    """
    Gera regiões de busca circulares em torno do bbox anterior,
    onde step_size define o deslocamento em pixels reais
    (1 px = step_size=1).
    """

    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2

    raio_max = (max(w, h) / 2) * search_region_scale

    yield (x, y, w, h)  # primeiro retorna o bbox original

    # passo radial em pixels
    for raio in np.arange(step_size, raio_max + step_size, step_size):
        circunferencia = 2 * np.pi * raio

        # define espaçamento angular de forma que os pontos na borda
        # fiquem separados por ~step_size pixels ao longo da circunferência
        num_steps = max(8, int(np.ceil(circunferencia / step_size)))
        direction = -1 if clockwise else 1

        thetas = start_angle + direction * np.linspace(0, 2 * np.pi, num_steps, endpoint=False)
        
        # deslocamentos em pixels
        pxs = (center_x + raio * np.cos(thetas) - w / 2).astype(int)
        pys = (center_y + raio * np.sin(thetas) - h / 2).astype(int)

        # garante que o bbox não ultrapasse os limites do frame
        pxs = np.clip(pxs, 0, frame_shape[1] - w)
        pys = np.clip(pys, 0, frame_shape[0] - h)

        for px, py in zip(pxs, pys):
            yield (px, py, w, h)


### Gerando regiões para o primeiro frame

In [ ]:
# Parâmetros
search_window_scale = parameters['MAX_SEARCH_WINDOW_SCALE']
step_size = parameters['STEP_SIZE']          # quanto menor -> mais granular

search_gen = generate_search_regions_circular(
    # prev_bbox = (16.0, 30.0, 34.0, 39.0), TIGER2
    prev_bbox = (122.0, 58.0, 75.0, 97.0), # DAVID
    # prev_bbox = (142.0, 62.0, 62.0, 98.0),
    frame_shape = first_frame.shape,
    search_region_scale = search_window_scale,
    step_size = step_size
)

# Tracker

In [ ]:
from collections import deque
import cv2
import numpy as np
import matplotlib.pyplot as plt
from wisardpkg import ClusWisard

print("\n🚀 Iniciando tracking sobre todos os frames...")

bg_model = AdaptiveBackgroundModel(
    alpha=parameters.get("BG_ALPHA", 0.9)
)
bg_model.initialize(first_frame)

# ============================================================
# PARÂMETROS ClusWisard
# ============================================================
CLUSWISARD_DISCRIMINATOR_LIMIT = parameters["CLUSWISARD_DISCRIMINATOR_LIMIT"]
CLUSWISARD_MIN_SCORE = parameters["CLUSWISARD_MIN_SCORE"]

# ============================================================
# FILA FIFO DE DISCRIMINADORES ADAPTATIVOS
# ============================================================
discriminator_queue = deque(maxlen=CLUSWISARD_DISCRIMINATOR_LIMIT)

# ============================================================
# FUNÇÃO DE REINSTÂNCIA DO MODELO
# ============================================================
def new_clus():
    return ClusWisard(
        parameters["CLUSWISARD_ADDRESS_SIZE"],
        parameters["CLUSWISARD_MIN_SCORE"],
        parameters["CLUSWISARD_THRESHOLD"],
        parameters["CLUSWISARD_DISCRIMINATOR_LIMIT"],
        bleachingActivated=parameters["CLUSWISARD_BLEACHING_ACTIVATED"],
        returnActivationDegree=parameters["CLUSWISARD_ACTIVATION_DEGREE"],
        returnConfidence=parameters["CLUSWISARD_RETURN_CONFIDENCE"],
        returnClassesDegrees=parameters["CLUSWISARD_CLASSES_DEGREES"],
    )
# ============================================================
# PATCH ÂNCORA (FRAME 0)
# ============================================================
first_frame = cv2.imread(image_paths[0])
first_frame = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)

first_patch = tracker_utils.extract_patch(first_frame, first_gt)
first_pattern = preprocess_patch(first_patch)

# ============================================================
# INICIALIZA ClusWisard
# ============================================================
clus = new_clus()
clus.train([first_pattern.tolist()], ["object"])

prev_bbox = first_gt
prev_frame = first_frame.copy()

all_predictions = [prev_bbox]

# --- ACUMULADORES DE RETREINO ---
retrain_patches = []
retrain_info = []

# ============================================================
# LOOP PRINCIPAL
# ============================================================
for frame_idx, frame_path in enumerate(image_paths[1:462], start=1):

    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"❌ Frame {frame_idx} não carregado!")
        all_predictions.append(prev_bbox)
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    print(f"\n🖼️ Frame {frame_idx}: {frame_path}")

    # --------------------------------------------------------
    # PASSO EXTRA: REMOÇÃO DE FUNDO GUIADA PELO FRAME ANTERIOR
    # --------------------------------------------------------
    if frame_idx==1:
        foreground = adaptive_background_subtraction(
        frame=frame,
        prev_bbox=prev_bbox,
        bg_model=bg_model,
        tracking_confidence=0.4,
        confidence_threshold=CLUSWISARD_MIN_SCORE,
        )
    else:
        
        foreground = adaptive_background_subtraction(
            frame=frame,
            prev_bbox=prev_bbox,
            bg_model=bg_model,
            tracking_confidence=best_region["activation"],
            confidence_threshold=CLUSWISARD_MIN_SCORE,
        )

    # --------------------------------------------------------
    # PASSO 1: REGIÕES DE BUSCA
    # --------------------------------------------------------
    search_gen = generate_search_regions_circular(
        prev_bbox=prev_bbox,
        frame_shape=foreground.shape,
        search_region_scale=parameters["MAX_SEARCH_WINDOW_SCALE"],
        step_size=parameters["STEP_SIZE"],
    )

    all_regions = list(search_gen)
    print(f"🔍 Geradas {len(all_regions)} regiões")

    # --------------------------------------------------------
    # PASSO 2: CLASSIFICAÇÃO
    # --------------------------------------------------------
    results = []

    for i, region in enumerate(all_regions):

        patch_region = tracker_utils.extract_patch(foreground, region)

        if patch_region.size == 0:
            continue

        pattern_region = preprocess_patch(patch_region)

        result = clus.classify([pattern_region.tolist()])[0]
        activation = result.get("activationDegree", -1.0)

        results.append(
            {
                "region_id": i,
                "bbox": region,
                "activation": activation,
                "patch": pattern_region,
                "total": result,
            }
        )

    if not results:
        print("⚠️ Nenhuma região válida — mantendo bbox anterior.")
        all_predictions.append(prev_bbox)
        prev_frame = foreground.copy()
        continue

    # --------------------------------------------------------
    # PASSO 3: MELHOR REGIÃO
    # --------------------------------------------------------
    best_region = max(results, key=lambda r: r["activation"])

    prev_bbox = best_region["bbox"]
    all_predictions.append(prev_bbox)

    print(
        f"🏆 Melhor região: {best_region['bbox']} "
        f"| ativação={best_region['activation']:.4f}"
    )

    # --------------------------------------------------------
    # VISUALIZAÇÃO
    # --------------------------------------------------------
    frame_vis = foreground.copy()

    for r in results:
        x, y, w, h = map(int, r["bbox"])
        cv2.rectangle(frame_vis, (x, y), (x + w, y + h), (255, 0, 0), 1)

    bx, by, bw, bh = map(int, best_region["bbox"])
    cv2.rectangle(frame_vis, (bx, by), (bx + bw, by + bh), (0, 255, 0), 3)

    best_patch = tracker_utils.extract_patch(foreground, best_region["bbox"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(frame_vis)
    axes[0].axis("off")
    axes[1].imshow(best_patch)
    axes[1].axis("off")
    plt.show()

    # --------------------------------------------------------
    # RETREINO BASEADO EM FILA
    # --------------------------------------------------------
    if best_region["activation"] <= CLUSWISARD_MIN_SCORE:

        print(
            f"🔁 Ativação baixa ({best_region['activation']:.4f}) "
            "→ adicionando discriminador adaptativo"
        )

        discriminator_queue.append(
            ([best_region["patch"].tolist()], ["object"])
        )

        retrain_patches.append(best_patch)
        retrain_info.append(
            {
                "frame": frame_idx,
                "activation": best_region["activation"],
                "bbox": best_region["bbox"],
            }
        )

        print("\n♻️ RETREINANDO ClusWisard DO ZERO")

        clus = new_clus()

        X_train = [first_pattern.tolist()]
        y_train = ["object"]

        for x, y in discriminator_queue:
            X_train.extend(x)
            y_train.extend(y)

        clus.train(X_train, y_train)

    # --------------------------------------------------------
    # ATUALIZA FRAME ANTERIOR
    # --------------------------------------------------------
    prev_frame = frame.copy()

# ============================================================
# FINAL
# ============================================================
print("\n✅ Tracking concluído!")
print(f"🔁 Total de retreinos: {len(retrain_patches)}")


In [ ]:
import cv2
import os
import shutil


# --- Pasta de saída ---
output_dir = "resultados/"

# Apaga a pasta se existir
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Cria nova pasta vazia
os.makedirs(output_dir, exist_ok=True)
# --- Salvar cada frame com a bbox prevista ---
for frame_idx, frame_path in enumerate(image_paths[1:]):
    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"Frame {frame_idx} não carregado!")
        continue

    # Pega a previsão correspondente
    if frame_idx < len(all_predictions):
        x, y, bw, bh = map(int, all_predictions[frame_idx])
        cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
        cv2.putText(frame, f"Frame {frame_idx}", (x, max(20, y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Salva imagem
    output_path = os.path.join(output_dir, f"frame_{frame_idx:04d}.png")
    cv2.imwrite(output_path, frame)

print(f"{len (all_predictions)} frames salvos em: {output_dir}")


In [ ]:
len(retrain_patches)

In [ ]:
##### import cv2
import os
import shutil

# --- Pasta de saída ---
# Apaga a pasta se existir
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# --- Configuração do vídeo ---
fps = 10  # ajuste se seu vídeo original tiver outro FPS

# Lê o primeiro frame para pegar o tamanho
first_frame = cv2.imread(image_paths[1])
h, w = first_frame.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(video_path, fourcc, fps, (w, h))

# --- Processar e gravar vídeo ---
for frame_idx, frame_path in enumerate(image_paths[1:364], start=1):

    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"Frame {frame_idx} não carregado!")
        continue

    # Pega a bbox prevista correspondente
    if frame_idx < len(all_predictions):
        x, y, bw, bh = map(int, all_predictions[frame_idx])
        cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"Frame {frame_idx}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2,
        )

    # Escreve no vídeo
    video_writer.write(frame)

video_writer.release()
print(f" Vídeo salvo com sucesso em: {video_path}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def bbox_center(bbox):
    """
    bbox: (x, y, w, h)
    retorna: (cx, cy)
    """
    x, y, w, h = bbox
    return np.array([x + w / 2.0, y + h / 2.0])


errors = []
frames = []

for i in range(0, 462, 5):

    frame_idx = i
    gt_bbox = all_ground_truths[i]

    result_bbox = all_predictions[i]

    gt_center = bbox_center(gt_bbox)
    res_center = bbox_center(result_bbox)

    error = np.linalg.norm(gt_center - res_center)

    errors.append(error)
    frames.append(frame_idx)

# --- PLOT ---
plt.figure()
plt.plot(frames, errors, linestyle='--')
plt.xlabel("Frame")
plt.ylabel("Erro de centro (pixels)")
plt.title("Erro entre centro do GT e centro do tracker")
plt.grid(True)
plt.show()

media = sum(errors) / len(errors)
print(media)


In [ ]:
# adicionar custo de remoção de fundo

In [ ]:
print(len(all_predictions))